# Movie Rental Data Warehouse — ETL Pipeline

This notebook implements a full **Extract → Transform → Load (ETL)** pipeline that:

1. Reads raw transactional data from the **Sakila** source database
2. Transforms each table into a clean dimension or fact table
3. Loads the results into the **`movie_rental_dw`** data warehouse

---

### Data Warehouse Schema

| Layer | Tables |
|-------|--------|
| **Dimensions** | `dim_date`, `dim_customer`, `dim_film`, `dim_store`, `dim_staff`, `dim_actor` |
| **Bridge** | `bridge_film_actor` |
| **Facts** | `fact_rental`, `fact_payment`, `fact_inventory_snapshot` |

---

## 1. Setup

Install the required Python packages if they are not already present.

In [1]:
 %pip install pandas sqlalchemy pymysql matplotlib

Note: you may need to restart the kernel to use updated packages.


## 2. Database Connections

Create SQLAlchemy engine connections for both databases:

- **`source_engine`** — points to `sakila`, the normalised OLTP database
- **`target_engine`** — points to `movie_rental_dw`, the data warehouse

> **Note:** Update `USER`, `PASSWORD`, `HOST`, and `PORT` to match your environment before running.

In [2]:
import pandas as pd
from sqlalchemy import create_engine

# =========================
# Database Connections
# =========================

SOURCE_DB = "sakila"
TARGET_DB = "movie_rental_dw"

USER = "root"
PASSWORD = "NewPassword123!"
HOST = "localhost"
PORT = "3306"

source_engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{SOURCE_DB}"
)

target_engine = create_engine(
    f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{TARGET_DB}"
)

## 3. Extract

Pull every source table from `sakila` into a Pandas DataFrame. These raw DataFrames are the input for all subsequent transform steps.

| DataFrame | Source Table | Purpose |
|-----------|-------------|----------|
| `rental` | `rental` | Rental transactions |
| `payment` | `payment` | Payment records |
| `customer` | `customer` | Customer master |
| `film` | `film` | Film catalogue |
| `inventory` | `inventory` | Physical inventory |
| `store` | `store` | Store locations |
| `staff` | `staff` | Staff roster |
| `address` / `city` / `country` | — | Geography lookups |
| `language` | `language` | Film language |
| `category` / `film_category` | — | Film genre mapping |
| `actor` / `film_actor` | — | Cast mapping |

In [3]:
# =========================
# Extract
# =========================

rental = pd.read_sql("SELECT * FROM rental", source_engine)
payment = pd.read_sql("SELECT * FROM payment", source_engine)
customer = pd.read_sql("SELECT * FROM customer", source_engine)
film = pd.read_sql("SELECT * FROM film", source_engine)
inventory = pd.read_sql("SELECT * FROM inventory", source_engine)
store = pd.read_sql("SELECT * FROM store", source_engine)
staff = pd.read_sql("SELECT * FROM staff", source_engine)
address = pd.read_sql("SELECT * FROM address", source_engine)
city = pd.read_sql("SELECT * FROM city", source_engine)
country = pd.read_sql("SELECT * FROM country", source_engine)
language = pd.read_sql("SELECT * FROM language", source_engine)
category = pd.read_sql("SELECT * FROM category", source_engine)
film_category = pd.read_sql("SELECT * FROM film_category", source_engine)
actor = pd.read_sql("SELECT * FROM actor", source_engine)
film_actor = pd.read_sql("SELECT * FROM film_actor", source_engine)

print("Extract completed.")

Extract completed.


## 4. Transform — `dim_date`

Build a **date dimension** that covers every date appearing in the rental or payment data, plus today's date.

**Steps:**
1. Parse `rental_date`, `return_date`, and `payment_date` as proper datetimes.
2. Collect all unique dates from those three columns into a single de-duplicated series.
3. Derive calendar attributes: day, month, quarter, year, day-of-week.
4. Generate an integer `date_key` in `YYYYMMDD` format for fast joins.
5. Prepend a sentinel **Unknown** row (`date_key = 0`) to handle NULLable foreign keys.

In [4]:
# =========================
# Transform dim_date
# =========================

rental["rental_date"] = pd.to_datetime(rental["rental_date"])
rental["return_date"] = pd.to_datetime(rental["return_date"])
payment["payment_date"] = pd.to_datetime(payment["payment_date"])

date_values = pd.concat([
    rental["rental_date"],
    rental["return_date"].dropna(),
    payment["payment_date"],
    pd.Series([pd.Timestamp.today().normalize()])
]).dropna().dt.date.drop_duplicates()

dim_date = pd.DataFrame({"full_date": pd.to_datetime(date_values)})

dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["day_number"] = dim_date["full_date"].dt.day
dim_date["month_number"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["quarter_number"] = dim_date["full_date"].dt.quarter
dim_date["year_number"] = dim_date["full_date"].dt.year
dim_date["day_of_week"] = dim_date["full_date"].dt.day_name()
dim_date["is_unknown"] = 0

unknown_date = pd.DataFrame([{
    "date_key": 0,
    "full_date": pd.NaT,
    "day_number": None,
    "month_number": None,
    "month_name": "Unknown",
    "quarter_number": None,
    "year_number": None,
    "day_of_week": "Unknown",
    "is_unknown": 1
}])

dim_date = pd.concat([unknown_date, dim_date], ignore_index=True)

## 5. Transform — `dim_customer`

Denormalise the customer geography hierarchy (`customer → address → city → country`) into a single flat row per customer.

**Steps:**
1. Select only the needed columns from each source table — this avoids duplicate `last_update` columns that would break the merge.
2. Inner-join `customer` → `address` → `city` → `country` in sequence.
3. Derive `full_name` by concatenating `first_name` and `last_name`.
4. Select the final set of columns for the dimension.

In [5]:
# =========================
# Transform dim_customer
# =========================

print("=== Source samples before dim_customer transform ===")

print("\nCustomer sample:")
print(customer[["customer_id", "first_name", "last_name", "email", "active", "address_id", "store_id", "create_date"]].head())

print("\nAddress sample:")
print(address[["address_id", "address", "district", "city_id", "postal_code", "phone"]].head())

print("\nCity sample:")
print(city[["city_id", "city", "country_id"]].head())

print("\nCountry sample:")
print(country[["country_id", "country"]].head())


# Select only needed columns to avoid duplicate last_update columns
customer_clean = customer[[
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "active",
    "address_id",
    "store_id",
    "create_date"
]]

address_clean = address[[
    "address_id",
    "address",
    "district",
    "city_id",
    "postal_code",
    "phone"
]]

city_clean = city[[
    "city_id",
    "city",
    "country_id"
]]

country_clean = country[[
    "country_id",
    "country"
]]


print("\n=== Step 1: Merge customer with address ===")
customer_address = customer_clean.merge(
    address_clean,
    on="address_id",
    how="inner"
)

print("Rows after customer + address:", customer_address.shape)
print(customer_address.head())


print("\n=== Step 2: Merge with city ===")
customer_address_city = customer_address.merge(
    city_clean,
    on="city_id",
    how="inner"
)

print("Rows after adding city:", customer_address_city.shape)
print(customer_address_city.head())


print("\n=== Step 3: Merge with country ===")
dim_customer = customer_address_city.merge(
    country_clean,
    on="country_id",
    how="inner"
)

print("Rows after adding country:", dim_customer.shape)
print(dim_customer.head())


print("\n=== Step 4: Create full_name ===")
dim_customer["full_name"] = (
    dim_customer["first_name"] + " " + dim_customer["last_name"]
)

print(dim_customer[["customer_id", "first_name", "last_name", "full_name"]].head())


print("\n=== Step 5: Select final dim_customer columns ===")
dim_customer = dim_customer[[
    "customer_id",
    "full_name",
    "email",
    "active",
    "address",
    "district",
    "city",
    "country",
    "postal_code",
    "phone",
    "store_id",
    "create_date"
]]

print("Final dim_customer shape:", dim_customer.shape)
print(dim_customer.head())

=== Source samples before dim_customer transform ===

Customer sample:
   customer_id first_name last_name                                email  \
0            1       MARY     SMITH        MARY.SMITH@sakilacustomer.org   
1            2   PATRICIA   JOHNSON  PATRICIA.JOHNSON@sakilacustomer.org   
2            3      LINDA  WILLIAMS    LINDA.WILLIAMS@sakilacustomer.org   
3            4    BARBARA     JONES     BARBARA.JONES@sakilacustomer.org   
4            5  ELIZABETH     BROWN   ELIZABETH.BROWN@sakilacustomer.org   

   active  address_id  store_id         create_date  
0       1           5         1 2006-02-14 22:04:36  
1       1           6         1 2006-02-14 22:04:36  
2       1           7         1 2006-02-14 22:04:36  
3       1           8         2 2006-02-14 22:04:36  
4       1           9         1 2006-02-14 22:04:36  

Address sample:
   address_id               address  district  city_id postal_code  \
0           1     47 MySakila Drive   Alberta      300       

## 6. Transform — `dim_film`

Enrich each film with its spoken language and genre categories.

**Steps:**
1. Join `film` with `language` to resolve the language name.
2. Resolve the many-to-many `film_category` bridge into a single comma-separated `category_name` string per film.
3. Left-join the category string onto the film+language result so films without a category are still included.
4. Strip whitespace from `language_name` and keep only the required columns.

In [6]:
# =========================
# Transform dim_film
# =========================

film_lang = film.merge(language, on="language_id")
film_cat = (
    film_category
    .merge(category, on="category_id")
    .groupby("film_id")["name"]
    .apply(lambda x: ", ".join(sorted(x.unique())))
    .reset_index()
    .rename(columns={"name": "category_name"})
)

dim_film = film_lang.merge(film_cat, on="film_id", how="left")

dim_film = dim_film.rename(columns={"name": "language_name"})

dim_film["language_name"] = dim_film["language_name"].str.strip()

dim_film = dim_film[[
    "film_id",
    "title",
    "description",
    "release_year",
    "rating",
    "rental_duration",
    "rental_rate",
    "length",
    "replacement_cost",
    "language_name",
    "category_name"
]]

## 7. Transform — `dim_store`

Denormalise store location data (`store → address → city → country`) into a single flat row per store.

The join pattern mirrors `dim_customer`: select clean column subsets first, then chain inner joins through the geography hierarchy.

In [7]:
# =========================
# Transform dim_store
# =========================

print("=== Source samples before dim_store transform ===")

print("\nStore sample:")
print(store[["store_id", "manager_staff_id", "address_id"]].head())

print("\nAddress sample:")
print(address[["address_id", "address", "district", "city_id", "postal_code", "phone"]].head())

print("\nCity sample:")
print(city[["city_id", "city", "country_id"]].head())

print("\nCountry sample:")
print(country[["country_id", "country"]].head())


# Select only needed columns to avoid duplicate last_update columns
store_clean = store[[
    "store_id",
    "manager_staff_id",
    "address_id"
]]

address_clean = address[[
    "address_id",
    "address",
    "district",
    "city_id",
    "postal_code",
    "phone"
]]

city_clean = city[[
    "city_id",
    "city",
    "country_id"
]]

country_clean = country[[
    "country_id",
    "country"
]]


print("\n=== Step 1: Merge store with address ===")
store_address = store_clean.merge(
    address_clean,
    on="address_id",
    how="inner"
)

print("Rows after store + address:", store_address.shape)
print(store_address.head())


print("\n=== Step 2: Merge with city ===")
store_address_city = store_address.merge(
    city_clean,
    on="city_id",
    how="inner"
)

print("Rows after adding city:", store_address_city.shape)
print(store_address_city.head())


print("\n=== Step 3: Merge with country ===")
dim_store = store_address_city.merge(
    country_clean,
    on="country_id",
    how="inner"
)

print("Rows after adding country:", dim_store.shape)
print(dim_store.head())


print("\n=== Step 4: Select final dim_store columns ===")
dim_store = dim_store[[
    "store_id",
    "manager_staff_id",
    "address",
    "district",
    "city",
    "country",
    "postal_code",
    "phone"
]]

print("Final dim_store shape:", dim_store.shape)
print(dim_store.head())

=== Source samples before dim_store transform ===

Store sample:
   store_id  manager_staff_id  address_id
0         1                 1           1
1         2                 2           2

Address sample:
   address_id               address  district  city_id postal_code  \
0           1     47 MySakila Drive   Alberta      300               
1           2    28 MySQL Boulevard       QLD      576               
2           3     23 Workhaven Lane   Alberta      300               
3           4  1411 Lillydale Drive       QLD      576               
4           5        1913 Hanoi Way  Nagasaki      463       35200   

         phone  
0               
1               
2  14033335568  
3   6172235589  
4  28303384290  

City sample:
   city_id                  city  country_id
0        1  A Coruña (La Coruña)          87
1        2                  Abha          82
2        3             Abu Dhabi         101
3        4                 Acuña          60
4        5                 Ada

## 8. Transform — `dim_staff`

Denormalise staff data (`staff → address → city → country`) into a flat dimension.

**Steps:**
1. Select clean column subsets from each source table.
2. Chain inner joins through the geography hierarchy.
3. Derive `full_name` from `first_name` + `last_name`.
4. Select the final columns for the dimension.

In [8]:
# =========================
# Transform dim_staff
# =========================

print("=== Source samples before dim_staff transform ===")

print("\nStaff sample:")
print(staff[[
    "staff_id", "first_name", "last_name", "email",
    "username", "active", "store_id", "address_id"
]].head())

print("\nAddress sample:")
print(address[[
    "address_id", "address", "city_id"
]].head())

print("\nCity sample:")
print(city[[
    "city_id", "city", "country_id"
]].head())

print("\nCountry sample:")
print(country[[
    "country_id", "country"
]].head())


# Select only needed columns to avoid duplicate last_update columns
staff_clean = staff[[
    "staff_id",
    "first_name",
    "last_name",
    "email",
    "username",
    "active",
    "store_id",
    "address_id"
]]

address_clean = address[[
    "address_id",
    "address",
    "city_id"
]]

city_clean = city[[
    "city_id",
    "city",
    "country_id"
]]

country_clean = country[[
    "country_id",
    "country"
]]


print("\n=== Step 1: Merge staff with address ===")
staff_address = staff_clean.merge(
    address_clean,
    on="address_id",
    how="inner"
)

print("Rows after staff + address:", staff_address.shape)
print(staff_address.head())


print("\n=== Step 2: Merge with city ===")
staff_address_city = staff_address.merge(
    city_clean,
    on="city_id",
    how="inner"
)

print("Rows after adding city:", staff_address_city.shape)
print(staff_address_city.head())


print("\n=== Step 3: Merge with country ===")
dim_staff = staff_address_city.merge(
    country_clean,
    on="country_id",
    how="inner"
)

print("Rows after adding country:", dim_staff.shape)
print(dim_staff.head())


print("\n=== Step 4: Create full_name ===")
dim_staff["full_name"] = (
    dim_staff["first_name"] + " " + dim_staff["last_name"]
)

print(dim_staff[["staff_id", "first_name", "last_name", "full_name"]].head())


print("\n=== Step 5: Select final dim_staff columns ===")
dim_staff = dim_staff[[
    "staff_id",
    "full_name",
    "email",
    "username",
    "active",
    "store_id",
    "address",
    "city",
    "country"
]]

print("Final dim_staff shape:", dim_staff.shape)
print(dim_staff.head())

=== Source samples before dim_staff transform ===

Staff sample:
   staff_id first_name last_name                         email username  \
0         1       Mike   Hillyer  Mike.Hillyer@sakilastaff.com     Mike   
1         2        Jon  Stephens  Jon.Stephens@sakilastaff.com      Jon   

   active  store_id  address_id  
0       1         1           3  
1       1         2           4  

Address sample:
   address_id               address  city_id
0           1     47 MySakila Drive      300
1           2    28 MySQL Boulevard      576
2           3     23 Workhaven Lane      300
3           4  1411 Lillydale Drive      576
4           5        1913 Hanoi Way      463

City sample:
   city_id                  city  country_id
0        1  A Coruña (La Coruña)          87
1        2                  Abha          82
2        3             Abu Dhabi         101
3        4                 Acuña          60
4        5                 Adana          97

Country sample:
   country_id      

## 9. Transform — `dim_actor`

A lightweight dimension — just `actor_id` and a derived `full_name`. No geography joins are needed here.

In [9]:
# =========================
# Transform dim_actor
# =========================

dim_actor = actor.copy()
dim_actor["full_name"] = dim_actor["first_name"] + " " + dim_actor["last_name"]

dim_actor = dim_actor[[
    "actor_id",
    "full_name"
]]

## 10. Clear Warehouse Tables

Truncate all target tables **in dependency order** (facts first, then bridge, then dimensions) so that foreign-key constraints do not block the truncation.

`FOREIGN_KEY_CHECKS` is temporarily disabled during the truncation and re-enabled immediately after.

> ⚠️ **This is a destructive operation.** All existing warehouse data will be deleted before the fresh load begins.

In [10]:
from sqlalchemy import text

tables_to_clear = [
    "fact_inventory_snapshot",
    "fact_payment",
    "fact_rental",
    "bridge_film_actor",
    "dim_actor",
    "dim_staff",
    "dim_store",
    "dim_film",
    "dim_customer",
    "dim_date"
]

with target_engine.begin() as conn:
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    
    for table in tables_to_clear:
        conn.execute(text(f"TRUNCATE TABLE {table};"))
        print(f"Cleared table: {table}")
    
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

print("All warehouse tables cleared. You can load dimensions again.")

Cleared table: fact_inventory_snapshot
Cleared table: fact_payment
Cleared table: fact_rental
Cleared table: bridge_film_actor
Cleared table: dim_actor
Cleared table: dim_staff
Cleared table: dim_store
Cleared table: dim_film
Cleared table: dim_customer
Cleared table: dim_date
All warehouse tables cleared. You can load dimensions again.


## 11. Load — Dimensions

Append all six dimension DataFrames to their corresponding warehouse tables using `if_exists='append'`.

Dimensions are loaded before facts and the bridge table because the fact and bridge rows reference dimension surrogate keys.

In [11]:
# =========================
# Load Dimensions
# =========================

dim_date.to_sql("dim_date", target_engine, if_exists="append", index=False)
dim_customer.to_sql("dim_customer", target_engine, if_exists="append", index=False)
dim_film.to_sql("dim_film", target_engine, if_exists="append", index=False)
dim_store.to_sql("dim_store", target_engine, if_exists="append", index=False)
dim_staff.to_sql("dim_staff", target_engine, if_exists="append", index=False)
dim_actor.to_sql("dim_actor", target_engine, if_exists="append", index=False)

print("Dimensions loaded.")

Dimensions loaded.


## 12. Load — `bridge_film_actor`

Resolve the many-to-many relationship between films and actors using the surrogate keys assigned by the warehouse.

**Steps:**
1. Re-read `film_key` and `actor_key` from the freshly loaded dimension tables.
2. Join the source `film_actor` table to both dimensions to replace natural keys with surrogate keys.
3. De-duplicate and load the resulting bridge table.

In [12]:
# =========================
# Load bridge_film_actor
# =========================

dw_dim_film = pd.read_sql("SELECT film_key, film_id FROM dim_film", target_engine)
dw_dim_actor = pd.read_sql("SELECT actor_key, actor_id FROM dim_actor", target_engine)

bridge_film_actor = (
    film_actor
    .merge(dw_dim_film, on="film_id")
    .merge(dw_dim_actor, on="actor_id")
)

bridge_film_actor = bridge_film_actor[[
    "film_key",
    "actor_key"
]].drop_duplicates()

bridge_film_actor.to_sql(
    "bridge_film_actor",
    target_engine,
    if_exists="append",
    index=False
)

print("Bridge table loaded.")

Bridge table loaded.


## 13. Transform & Load — `fact_rental`

Build the central rental fact table by joining the source `rental` table to every relevant dimension.

**Derived measures:**

| Column | Description |
|--------|-------------|
| `rental_count` | Always 1 — used for simple aggregation |
| `rental_duration_days` | Actual days the item was rented (`return_date − rental_date`) |
| `expected_rental_duration_days` | The contractual duration from `dim_film.rental_duration` |
| `late_return_flag` | `1` if the item was returned late, `0` if on time, `NULL` if not yet returned |
| `late_days` | Number of days overdue (0 if returned on time, `NULL` if not yet returned) |

In [13]:
# =========================
# Transform fact_rental
# =========================

dw_dim_customer = pd.read_sql("SELECT customer_key, customer_id FROM dim_customer", target_engine)
dw_dim_film = pd.read_sql("SELECT film_key, film_id, rental_duration FROM dim_film", target_engine)
dw_dim_store = pd.read_sql("SELECT store_key, store_id FROM dim_store", target_engine)
dw_dim_staff = pd.read_sql("SELECT staff_key, staff_id FROM dim_staff", target_engine)

fact_rental = (
    rental
    .merge(inventory[["inventory_id", "film_id", "store_id"]], on="inventory_id")
    .merge(dw_dim_customer, on="customer_id")
    .merge(dw_dim_film, on="film_id")
    .merge(dw_dim_store, on="store_id")
    .merge(dw_dim_staff, on="staff_id")
)

fact_rental["rental_date_key"] = fact_rental["rental_date"].dt.strftime("%Y%m%d").astype(int)

fact_rental["return_date_key"] = (
    fact_rental["return_date"]
    .dt.strftime("%Y%m%d")
    .fillna("0")
    .astype(int)
)

fact_rental["rental_count"] = 1

fact_rental["rental_duration_days"] = (
    fact_rental["return_date"] - fact_rental["rental_date"]
).dt.days

fact_rental["expected_rental_duration_days"] = fact_rental["rental_duration"]

fact_rental["late_return_flag"] = None
returned_mask = fact_rental["return_date"].notna()

fact_rental.loc[returned_mask, "late_return_flag"] = (
    fact_rental.loc[returned_mask, "rental_duration_days"]
    > fact_rental.loc[returned_mask, "expected_rental_duration_days"]
).astype(int)

fact_rental["late_days"] = None
fact_rental.loc[returned_mask, "late_days"] = (
    fact_rental.loc[returned_mask, "rental_duration_days"]
    - fact_rental.loc[returned_mask, "expected_rental_duration_days"]
).clip(lower=0)

fact_rental = fact_rental[[
    "rental_id",
    "inventory_id",
    "rental_date_key",
    "return_date_key",
    "customer_key",
    "film_key",
    "store_key",
    "staff_key",
    "rental_count",
    "rental_duration_days",
    "expected_rental_duration_days",
    "late_return_flag",
    "late_days"
]]

fact_rental.to_sql("fact_rental", target_engine, if_exists="append", index=False)

print("fact_rental loaded.")

fact_rental loaded.


## 14. Transform & Load — `fact_payment`

Build the payment fact table by linking each payment to its customer, staff member, film, and store.

Because a payment references a rental which references an inventory item, the film and store are resolved transitively via `rental → inventory`.

**Derived measures:**

| Column | Description |
|--------|-------------|
| `payment_amount` | The amount paid (directly from `payment.amount`) |
| `payment_count` | Always 1 — used for simple aggregation |

In [14]:
# =========================
# Transform fact_payment
# =========================

fact_payment = (
    payment
    .merge(dw_dim_customer, on="customer_id")
    .merge(dw_dim_staff, on="staff_id")
    .merge(rental[["rental_id", "inventory_id"]], on="rental_id", how="left")
    .merge(inventory[["inventory_id", "film_id", "store_id"]], on="inventory_id", how="left")
    .merge(dw_dim_film[["film_key", "film_id"]], on="film_id", how="left")
    .merge(dw_dim_store, on="store_id", how="left")
)

fact_payment["payment_date_key"] = (
    fact_payment["payment_date"].dt.strftime("%Y%m%d").astype(int)
)

fact_payment["payment_amount"] = fact_payment["amount"]
fact_payment["payment_count"] = 1

fact_payment = fact_payment[[
    "payment_id",
    "rental_id",
    "payment_date_key",
    "customer_key",
    "film_key",
    "store_key",
    "staff_key",
    "payment_amount",
    "payment_count"
]]

fact_payment.to_sql("fact_payment", target_engine, if_exists="append", index=False)

print("fact_payment loaded.")

fact_payment loaded.


## 15. Transform & Load — `fact_inventory_snapshot`

Build a **point-in-time inventory snapshot** that shows, for each film×store combination, how many copies exist and how many are currently rented out.

**Steps:**
1. Identify open rentals (those with no `return_date`) to flag which inventory items are currently rented.
2. Left-join that flag onto the full inventory so every item appears, rented or not.
3. Resolve dimension surrogate keys for film and store.
4. Group by `film_key × store_key` and aggregate counts.
5. Compute `available_count = inventory_count − rented_count`.
6. Tag every row with today's date key as the snapshot date.

**Measures:**

| Column | Description |
|--------|-------------|
| `inventory_count` | Total physical copies of the film at that store |
| `rented_count` | Copies currently checked out |
| `available_count` | Copies available to rent right now |

In [15]:
# =========================
# Transform fact_inventory_snapshot
# =========================

open_rentals = (
    rental[rental["return_date"].isna()][["inventory_id"]]
    .drop_duplicates()
)

inventory_snapshot = inventory.merge(
    open_rentals.assign(is_rented=1),
    on="inventory_id",
    how="left"
)

inventory_snapshot["is_rented"] = inventory_snapshot["is_rented"].fillna(0).astype(int)

inventory_snapshot = (
    inventory_snapshot
    .merge(dw_dim_film[["film_key", "film_id"]], on="film_id")
    .merge(dw_dim_store, on="store_id")
)

fact_inventory_snapshot = (
    inventory_snapshot
    .groupby(["film_key", "store_key"])
    .agg(
        inventory_count=("inventory_id", "count"),
        rented_count=("is_rented", "sum")
    )
    .reset_index()
)

fact_inventory_snapshot["available_count"] = (
    fact_inventory_snapshot["inventory_count"]
    - fact_inventory_snapshot["rented_count"]
)

fact_inventory_snapshot["snapshot_date_key"] = int(pd.Timestamp.today().strftime("%Y%m%d"))

fact_inventory_snapshot = fact_inventory_snapshot[[
    "snapshot_date_key",
    "film_key",
    "store_key",
    "inventory_count",
    "rented_count",
    "available_count"
]]

fact_inventory_snapshot.to_sql(
    "fact_inventory_snapshot",
    target_engine,
    if_exists="append",
    index=False
)

print("fact_inventory_snapshot loaded.")

fact_inventory_snapshot loaded.


---

## ✅ ETL Complete

All dimension, bridge, and fact tables have been loaded into `movie_rental_dw`. The warehouse is ready for reporting and analytics.